# Task 4: Proxy Target Variable Using RFM Analysis

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [2]:
df = pd.read_csv("../data/raw/data.csv")

In [3]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

## RFM Feature Engineering

RFM features were created to capture customer transaction behavior based on recency, transaction frequency, and monetary value.

In [4]:
snapshot_date = df["TransactionStartTime"].max()

rfm = df.groupby("CustomerId").agg({
    "TransactionStartTime": lambda x: (
        snapshot_date - x.max()
    ).days,

    "TransactionId": "count",

    "Amount": "sum"
})

In [5]:
rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm.head()

,Recency,Frequency,Monetary
CustomerId,,,
CustomerId_1,83,1,-10000.0
CustomerId_10,83,1,-10000.0
CustomerId_1001,89,5,20000.0
CustomerId_1002,25,11,4225.0
CustomerId_1003,11,6,20000.0


In [6]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

In [7]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

In [8]:
rfm["Cluster"].value_counts()

Cluster
2    2314
0    1427
1       1
Name: count, dtype: int64

In [9]:
rfm["is_high_risk"] = (
    rfm["Cluster"] == 0
).astype(int)

In [10]:
rfm["is_high_risk"].value_counts()

is_high_risk
0    2315
1    1427
Name: count, dtype: int64

In [11]:
rfm.to_csv(
    "../data/processed/rfm_features.csv"
)

## Interpretation

Customer behavioral segments were generated using RFM analysis and KMeans clustering. The resulting clusters were used to create a proxy target variable representing potential high-risk customers for downstream credit risk modeling.